# R8 — GenAI claims adjudication: rule-based verifier vs LLM (ai_query)

Adjudicates the 20 probe claims from `adjudication_probe_claims.csv` two independent ways and measures agreement:
1. **`claims.genai.adjudication_probes`** — the 20 claim ids from the CSV.
2. **`claims.genai.adjudication_verifier`** — a deterministic rules engine (policy docs 01/02/03/04/05) emitting the 7 required typed columns per claim.
3. **`claims.genai.adjudications`** — the LLM (Llama-3.3-70B via the Databricks SQL `ai_query()` function, temperature 0, JSON `responseFormat`, parsed with `from_json`) emits the same 7 columns + `reasoning`. The query lives in `sql/r8_adjudications.sql`, which this notebook keeps in sync and executes.
4. **Agreement ≥ 90%** of the 20 claims must match the verifier on all six decision columns (`payable`, `waiting_period_ok`, `network_at_admission`, `room_rent_deduction_pct`, `copay_pct`, `reason_code`), and every trap-style claim (waiting-period violation, PED violation, non-network, room-rent over cap) must agree.

User-only evidence: screenshot `evidence/screenshots/r8_adjudications.png`.

Policy rules applied (verbatim from the volume docs): individual plans 30-day initial / 36-month PED waiting, 0% copay; SENIOR 30-day initial / 24-month PED, 10% copay; CORP_GROUP 0-day initial / 12-month PED, 0% copay; non-network adds 20% copay but stays payable; room rent above cap reduces the whole bill by (1 − cap/rent)×100, rounded to 2 decimals.

In [0]:
# Paper 4 config cell - same constants as R4/R5/R6/R7
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"   # adjudicator LLM (ai_query endpoint)
EMBED_MODEL = "databricks-gte-large-en"                 # Paper 4 embed model (config rule)
TEMPERATURE = 0.0                                       # every LLM call must be deterministic

PROBES_CSV = "/Volumes/claims/raw/landing/genai/adjudication_probe_claims.csv"
PROBES_TABLE = f"{CATALOG}.genai.adjudication_probes"
VERIFIER_TABLE = f"{CATALOG}.genai.adjudication_verifier"
ADJUDICATIONS_TABLE = f"{CATALOG}.genai.adjudications"
FACTS_TABLE = f"{CATALOG}.gold.claims_enriched"
SQL_NEEDLE_FILE = "/Workspace/Users/custom001x@gmail.com/saffron-shield/sql/r8_adjudications.sql"

REASON_CODES = {"INITIAL_WAITING", "PED_WAITING", "PAYABLE_WITH_DEDUCTION", "PAYABLE_IN_FULL"}
AGREEMENT_TARGET = 0.90   # required: >= 90% of probe claims agree with the verifier

print(f"CHAT_MODEL={CHAT_MODEL} | EMBED_MODEL={EMBED_MODEL} | temperature={TEMPERATURE}")
print(f"tables: {PROBES_TABLE} / {VERIFIER_TABLE} / {ADJUDICATIONS_TABLE}")
print(f"sql needle file: {SQL_NEEDLE_FILE}")

In [0]:
# 1. claims.genai.adjudication_probes: exactly the claim ids of adjudication_probe_claims.csv.
# Claim FACTS (plan, tenure, network, room rent, dates) are joined from claims.gold.claims_enriched.
spark.sql(f"""
CREATE OR REPLACE TABLE {PROBES_TABLE} AS
SELECT CAST(claim_id AS STRING) AS claim_id
FROM read_files('{PROBES_CSV}', format => 'csv', header => 'true')
""")

_probes = spark.table(PROBES_TABLE)
_csv_ids = {r.claim_id for r in spark.read.option("header", "true").csv(PROBES_CSV).collect()}
assert _probes.count() == 20, _probes.count()
assert {r.claim_id for r in _probes.collect()} == _csv_ids, "probes table != csv ids"

# every probe claim must have adjudication facts in the gold table (20/20 expected)
_missing = spark.sql(f"""
  SELECT p.claim_id FROM {PROBES_TABLE} p
  LEFT JOIN {FACTS_TABLE} f ON p.claim_id = f.claim_id
  WHERE f.claim_id IS NULL""").collect()
assert not _missing, f"probe claims missing from {FACTS_TABLE}: {[r.claim_id for r in _missing]}"

print(f"{PROBES_TABLE}: {_probes.count()} probe claims, all with facts in {FACTS_TABLE}")
display(_probes.orderBy("claim_id"))

In [0]:
# 2. claims.genai.adjudication_verifier: the deterministic rules engine (the answer key).
#    Rules verbatim from the policy docs (waiting periods 02/03, copay 02+05, room rent 04):
#      initial waiting: individual+senior 30 days, group 0 days
#      PED waiting: individual 36m, senior 24m, group 12m (completed tenure months at admission)
#      plan copay: senior 10%, others 0%; non-network adds 20% (claim stays payable)
#      room rent above cap: whole-bill deduction (1 - cap/rent)*100 rounded to 2 decimals
#    reason_code priority: INITIAL_WAITING (admission inside initial waiting)
#      > PED_WAITING (PED, tenure below PED months) > PAYABLE_WITH_DEDUCTION (deduction > 0)
#      > PAYABLE_IN_FULL. room_rent_deduction_pct is a property of the claim (computed whenever
#      room rent exceeds the cap) even when the claim is not payable.
spark.sql(f"""
CREATE OR REPLACE TABLE {VERIFIER_TABLE} AS
WITH facts AS (
  SELECT
    p.claim_id,
    c.plan_family,
    c.plan_code_at_admission,
    c.is_pre_existing,
    c.policy_tenure_months,
    DATEDIFF(c.admission_date, c.policy_start_date) AS days_since_policy_start,
    c.is_network_at_admission,
    c.room_rent_per_day,
    c.room_rent_cap_per_day
  FROM {PROBES_TABLE} p
  INNER JOIN {FACTS_TABLE} c ON p.claim_id = c.claim_id
),
rules AS (
  SELECT *,
    CASE WHEN plan_family = 'group' THEN 0 ELSE 30 END AS initial_days,
    CASE WHEN plan_family = 'group' THEN 12
         WHEN plan_code_at_admission = 'SENIOR' THEN 24
         ELSE 36 END AS ped_months,
    CASE WHEN plan_code_at_admission = 'SENIOR' THEN 10 ELSE 0 END AS plan_copay_pct
  FROM facts
),
decided AS (
  SELECT *,
    days_since_policy_start >= initial_days
      AND (NOT is_pre_existing OR policy_tenure_months >= ped_months) AS waiting_period_ok
  FROM rules
)
SELECT
  claim_id,
  waiting_period_ok AS payable,
  waiting_period_ok AS waiting_period_ok,
  is_network_at_admission AS network_at_admission,
  CAST(CASE WHEN room_rent_per_day > room_rent_cap_per_day
            THEN ROUND((1 - room_rent_cap_per_day / room_rent_per_day) * 100, 2)
            ELSE 0.00 END AS DOUBLE) AS room_rent_deduction_pct,
  CAST(plan_copay_pct + CASE WHEN is_network_at_admission THEN 0 ELSE 20 END AS INT) AS copay_pct,
  CASE
    WHEN NOT waiting_period_ok AND days_since_policy_start < initial_days THEN 'INITIAL_WAITING'
    WHEN NOT waiting_period_ok THEN 'PED_WAITING'
    WHEN room_rent_per_day > room_rent_cap_per_day THEN 'PAYABLE_WITH_DEDUCTION'
    ELSE 'PAYABLE_IN_FULL'
  END AS reason_code
FROM decided
ORDER BY claim_id
""")

_ver = spark.table(VERIFIER_TABLE)
assert _ver.count() == 20
import pyspark.sql.functions as F
_rc = {r["reason_code"] for r in _ver.select("reason_code").collect()}
assert _rc <= REASON_CODES, _rc
print(f"{VERIFIER_TABLE}: 20 rows | reason codes: {sorted(_rc)}")
print("verifier distribution:")
display(_ver.groupBy("reason_code", "payable").count().orderBy("reason_code"))
display(_ver.orderBy("claim_id"))

In [0]:
# 3. claims.genai.adjudications: the LLM adjudicator via ai_query().
#    The query is generated here and written to sql/r8_adjudications.sql (code needle: ai_query +
#    JSON responseFormat, parsed with from_json), so the file and the executed SQL cannot drift.
#    temperature = 0 on every call. If a row fails to parse, a repair pass re-queries ONLY those
#    claim ids without responseFormat and parses the raw reply with from_json(regexp_extract(...))
#    - the permitted fallback needle.
import re
from pathlib import Path

SCHEMA_DDL = ("payable BOOLEAN, waiting_period_ok BOOLEAN, network_at_admission BOOLEAN, "
              "room_rent_deduction_pct DOUBLE, copay_pct INT, reason_code STRING, reasoning STRING")

RULES_TEXT = (
    "You are a Saffron Shield claims adjudicator. Apply ONLY the rules below to the claim facts and decide the claim. "
    "RULE 1 - waiting periods. Individual plans (SILVER_IND and GOLD_IND) have a 30 day initial waiting period and a 36 month pre-existing disease waiting period. The SENIOR plan has a 30 day initial waiting period, a 24 month pre-existing disease waiting period and a 10 percent plan copay. CORP_GROUP has no initial waiting period (0 days) and a 12 month pre-existing disease waiting period. Months are completed months of policy tenure at admission. If the admission date is within the initial waiting period the claim is not payable and the reason is INITIAL_WAITING. Otherwise, if the claim is for a pre-existing disease and the policy tenure is below the PED waiting period, the claim is not payable and the reason is PED_WAITING. Compare the policy tenure months directly with the PED waiting months; for CORP_GROUP a pre-existing disease claim with tenure below 12 months is not payable with reason PED_WAITING, even if the tenure is more than half of the waiting period. "
    "RULE 2 - network. A claim at a non-network hospital is still payable but carries a 20 percent copayment added to the plan copay. The Senior plan copay is 10 percent; all other plans have a 0 percent plan copay. "
    "RULE 3 - room rent cap. The room rent deduction depends ONLY on room rent per day versus the plan cap. When room rent per day is above the plan cap, the whole bill is reduced by (1 minus cap divided by room rent per day) times 100, rounded to 2 decimals, and the reason becomes PAYABLE_WITH_DEDUCTION. When room rent is at or below the cap, the deduction is exactly 0 and the deduction reason must not be used. A non-network hospital never creates a deduction; it only adds 20 percent to copay_pct. room_rent_deduction_pct is a PERCENTAGE between 0 and 100: a 25 percent deduction is written 25.00, never 0.25. Example: room rent per day 4000 with a cap of 3000 gives room_rent_deduction_pct 25.00; room rent 6500 with a cap of 6000 gives 7.69. ALWAYS compute room_rent_deduction_pct from the room rent rule, even when the claim is NOT payable: a claim refused for a waiting period still reports the deduction its room rent would cause (example: a not-payable claim with room rent 4000 and cap 3000 reports room_rent_deduction_pct 25.00 and reason PED_WAITING or INITIAL_WAITING). "
    "RULE 4 - reason codes. A claim that fails a waiting period is not payable, with reason INITIAL_WAITING or PED_WAITING as set by rule 1. A payable claim with a room rent deduction above zero has reason PAYABLE_WITH_DEDUCTION. A payable claim with zero deduction has reason PAYABLE_IN_FULL, even when the hospital is non-network. Use these reason codes only: INITIAL_WAITING, PED_WAITING, PAYABLE_WITH_DEDUCTION, PAYABLE_IN_FULL. A claim that fails a waiting period MUST have payable=false and waiting_period_ok=false. "
    "Reply with ONLY a JSON object, no markdown and no extra text, with exactly these keys IN THIS ORDER: reasoning FIRST (work out the decision step by step here: check the initial waiting period, then the PED waiting period, then the network copay, then the room rent cap and its deduction in every case), THEN payable (boolean), waiting_period_ok (boolean), network_at_admission (boolean), room_rent_deduction_pct (number with 2 decimals), copay_pct (integer), reason_code (one of the four codes above). The decision fields MUST agree with your own reasoning. "
)

_FACTS_SQL = """WITH facts AS (
  SELECT
    p.claim_id,
    c.plan_family,
    c.plan_code_at_admission,
    c.is_pre_existing,
    c.policy_tenure_months,
    DATEDIFF(c.admission_date, c.policy_start_date) AS days_since_policy_start,
    c.is_network_at_admission,
    c.room_rent_per_day,
    c.room_rent_cap_per_day
  FROM claims.genai.adjudication_probes AS p
  INNER JOIN claims.gold.claims_enriched AS c
    ON p.claim_id = c.claim_id"""


def _prompt_expr():
    casts = {"is_pre_existing", "policy_tenure_months", "days_since_policy_start",
             "is_network_at_admission", "room_rent_per_day", "room_rent_cap_per_day"}
    facts = [("CLAIM FACTS: claim_id=", "claim_id"), ("; plan=", "plan_code_at_admission"),
             ("; plan_family=", "plan_family"), ("; is_pre_existing=", "is_pre_existing"),
             ("; policy_tenure_months=", "policy_tenure_months"),
             ("; days_since_policy_start=", "days_since_policy_start"),
             ("; is_network_at_admission=", "is_network_at_admission"),
             ("; room_rent_per_day=", "room_rent_per_day"),
             ("; room_rent_cap_per_day=", "room_rent_cap_per_day")]
    args = [f"'" + RULES_TEXT + "'"]
    for label, col in facts:
        val = f"CAST(f.{col} AS STRING)" if col in casts else f"f.{col}"
        args.append(f"'{label}'")
        args.append(val)
    return "CONCAT(\n      " + ",\n      ".join(args) + "\n    )"


_SELECT = """SELECT
  claim_id,
  CAST(j.payable AS BOOLEAN) AS payable,
  CAST(j.waiting_period_ok AS BOOLEAN) AS waiting_period_ok,
  CAST(j.network_at_admission AS BOOLEAN) AS network_at_admission,
  ROUND(CAST(j.room_rent_deduction_pct AS DOUBLE), 2) AS room_rent_deduction_pct,
  CAST(j.copay_pct AS INT) AS copay_pct,
  UPPER(TRIM(CAST(j.reason_code AS STRING))) AS reason_code,
  CAST(j.reasoning AS STRING) AS reasoning"""


def _build_primary():
    return f"""-- R8 - LLM claims adjudication via the Databricks SQL ai_query() function.
-- Endpoint = Paper 4 CHAT_MODEL ({CHAT_MODEL}), temperature = {TEMPERATURE} for determinism.
-- responseFormat (json_object) enforces JSON output; from_json(REGEXP_EXTRACT(...)) lands the
-- reply in typed columns. Generated and executed by GenAi-1/R8; this file is the code-needle artifact.
CREATE OR REPLACE TABLE {ADJUDICATIONS_TABLE} AS
{_FACTS_SQL}
),
prompts AS (
  SELECT
    f.*,
    {_prompt_expr()} AS prompt
  FROM facts AS f
),
replies AS (
  SELECT
    pr.*,
    AI_QUERY(
      '{CHAT_MODEL}',
      pr.prompt,
      modelParameters => named_struct('temperature', {TEMPERATURE}),
      responseFormat => '{{"type": "json_object"}}',
      failOnError => true
    ) AS reply
  FROM prompts AS pr
)
{_SELECT}
FROM (
  SELECT
    claim_id,
    FROM_JSON(REGEXP_EXTRACT(reply, '(?s)[{{].*[}}]', 0), '{SCHEMA_DDL}') AS j
  FROM replies
)
"""


def _build_retry(bad_ids):
    idlist = ", ".join(f"'{c}'" for c in bad_ids)
    return f"""-- R8 repair pass: re-adjudicate ONLY the rows whose first reply could not be parsed.
-- Permitted fallback: no responseFormat; the raw reply is JSON-extracted and parsed with from_json.
CREATE OR REPLACE TABLE {ADJUDICATIONS_TABLE} AS
{_FACTS_SQL}
  WHERE p.claim_id IN ({idlist})
),
prompts AS (
  SELECT
    f.*,
    {_prompt_expr()} AS prompt
  FROM facts AS f
),
replies AS (
  SELECT
    pr.*,
    AI_QUERY(
      '{CHAT_MODEL}',
      pr.prompt,
      modelParameters => named_struct('temperature', {TEMPERATURE}),
      failOnError => true
    ) AS reply
  FROM prompts AS pr
)
{_SELECT}
FROM (
  SELECT
    claim_id,
    FROM_JSON(REGEXP_EXTRACT(reply, '(?s)[{{].*[}}]', 0), '{SCHEMA_DDL}') AS j
  FROM replies
)
UNION ALL
SELECT claim_id, payable, waiting_period_ok, network_at_admission,
       room_rent_deduction_pct, copay_pct, reason_code, reasoning
FROM {ADJUDICATIONS_TABLE}
WHERE payable IS NOT NULL AND waiting_period_ok IS NOT NULL AND network_at_admission IS NOT NULL
  AND room_rent_deduction_pct IS NOT NULL AND copay_pct IS NOT NULL
  AND reason_code IN ('INITIAL_WAITING', 'PED_WAITING', 'PAYABLE_WITH_DEDUCTION', 'PAYABLE_IN_FULL')
"""


def _write_and_run(sql_text, expect_response_format):
    # code needles + config consistency, on the exact text being executed AND saved
    assert re.search(r"ai_query\s*\(", sql_text, re.I), "query lacks ai_query("
    assert re.search(r"from_json\s*\(", sql_text, re.I), "query lacks from_json parsing"
    assert f"'{CHAT_MODEL}'" in sql_text, "endpoint != config CHAT_MODEL"
    assert re.search(r"'temperature'\s*,\s*0(?:\.0+)?\s*\)", sql_text), "temperature != 0"
    if expect_response_format:
        assert re.search(r"responseFormat\s*=>", sql_text, re.I), "query lacks responseFormat"
    Path(SQL_NEEDLE_FILE).write_text(sql_text)
    spark.sql(sql_text)


def _bad_ids():
    _rows = spark.sql(f"""
      SELECT claim_id FROM {ADJUDICATIONS_TABLE}
      WHERE payable IS NULL OR waiting_period_ok IS NULL OR network_at_admission IS NULL
         OR room_rent_deduction_pct IS NULL OR copay_pct IS NULL OR reasoning IS NULL
         OR reason_code IS NULL
         OR reason_code NOT IN ('INITIAL_WAITING', 'PED_WAITING', 'PAYABLE_WITH_DEDUCTION', 'PAYABLE_IN_FULL')
    """).collect()
    return sorted(r.claim_id for r in _rows)


print(f"executing primary pass from {SQL_NEEDLE_FILE} (20 ai_query calls, temperature={TEMPERATURE}) ...")
_write_and_run(_build_primary(), expect_response_format=True)

_bad = _bad_ids()
if _bad:
    print(f"repair pass (permitted from_json fallback) for: {_bad}")
    _write_and_run(_build_retry(_bad), expect_response_format=False)
    _bad = _bad_ids()
assert not _bad, f"rows still unparseable after repair: {_bad}"

_adj = spark.table(ADJUDICATIONS_TABLE)
assert _adj.count() == 20, _adj.count()

import pyspark.sql.functions as F
_n_ok = _adj.filter(F.length("reasoning") >= 20).count()
print(f"{ADJUDICATIONS_TABLE}: 20 rows | reasoning >= 20 chars on {_n_ok}/20 rows (need >= 18)")
display(_adj.orderBy("claim_id"))

In [0]:
# 4. Agreement: >= 90% of the 20 claims must agree with the verifier on ALL six decision
#    columns (payable, waiting_period_ok, network_at_admission, room_rent_deduction_pct within
#    0.01, copay_pct, reason_code), and every trap-style claim must agree. Trap-style = a claim
#    that exercises any special rule: waiting-period violation, PED violation, non-network, or
#    room rent above the cap.
import pyspark.sql.functions as F

_v = {r["claim_id"]: r.asDict() for r in spark.table(VERIFIER_TABLE).collect()}
_a = {r["claim_id"]: r.asDict() for r in spark.table(ADJUDICATIONS_TABLE).collect()}
assert set(_v) == set(_a) and len(_v) == 20, sorted(set(_v) ^ set(_a))

_DECISION_COLS = ["payable", "waiting_period_ok", "network_at_admission", "room_rent_deduction_pct", "copay_pct", "reason_code"]


def _agrees(cid):
    v, a = _v[cid], _a[cid]
    return all([
        bool(v["payable"]) == bool(a["payable"]),
        bool(v["waiting_period_ok"]) == bool(a["waiting_period_ok"]),
        bool(v["network_at_admission"]) == bool(a["network_at_admission"]),
        abs(float(v["room_rent_deduction_pct"]) - float(a["room_rent_deduction_pct"])) <= 0.01,
        int(v["copay_pct"]) == int(a["copay_pct"]),
        v["reason_code"] == a["reason_code"],
    ])


_per_claim = []
for cid in sorted(_v):
    diffs = [c for c in _DECISION_COLS if not (
        (bool(_v[cid][c]) == bool(_a[cid][c])) if c in ("payable", "waiting_period_ok", "network_at_admission")
        else (int(_v[cid][c]) == int(_a[cid][c])) if c == "copay_pct"
        else (abs(float(_v[cid][c]) - float(_a[cid][c])) <= 0.01) if c == "room_rent_deduction_pct"
        else (_v[cid][c] == _a[cid][c]))]
    _per_claim.append({"claim_id": cid, "agree": _agrees(cid), "mismatched_columns": ",".join(diffs) or "-"})

_n_agree = sum(1 for p in _per_claim if p["agree"])
_agreement_rate = _n_agree / 20.0

# trap-style claims: special rules exercised -> must ALL agree (spec: every trap claim agrees)
_trap = [cid for cid in _v
         if not _v[cid]["waiting_period_ok"] or not _v[cid]["network_at_admission"]
         or float(_v[cid]["room_rent_deduction_pct"]) > 0]
_trap_bad = [cid for cid in _trap if not _agrees(cid)]

assert _agreement_rate >= AGREEMENT_TARGET, f"agreement {_agreement_rate:.2f} < {AGREEMENT_TARGET}"
assert not _trap_bad, f"trap-style claims disagree: {_trap_bad}"

print(f"agreement = {_n_agree}/20 = {_agreement_rate:.0%} (target >= {AGREEMENT_TARGET:.0%})")
print(f"trap-style claims ({len(_trap)}): {sorted(_trap)} - all agree")
display(spark.createDataFrame(_per_claim).orderBy("claim_id"))

In [0]:
# 5. Verify every R8 rule from the PERSISTED tables with fresh checks:
#    tables exist with the exact required columns and types, 20 rows each, reason codes in the
#    enum, reasoning >= 20 chars on >= 18 rows, agreement >= 90% recomputed in SQL, and both code
#    needles (ai_query + responseFormat/from_json) present in the sql/ file.
import json
import re
from pathlib import Path

# --- probes: exact CSV ids ---
_csv_ids = {r.claim_id for r in spark.read.option("header", "true").csv(PROBES_CSV).collect()}
_probe_ids = {r.claim_id for r in spark.table(PROBES_TABLE).collect()}
assert _probe_ids == _csv_ids and len(_probe_ids) == 20

# --- schemas: exact columns, exact types ---
_EXPECTED = {
    VERIFIER_TABLE: [("claim_id", "string"), ("payable", "boolean"), ("waiting_period_ok", "boolean"),
                     ("network_at_admission", "boolean"), ("room_rent_deduction_pct", "double"),
                     ("copay_pct", "int"), ("reason_code", "string")],
    ADJUDICATIONS_TABLE: [("claim_id", "string"), ("payable", "boolean"), ("waiting_period_ok", "boolean"),
                          ("network_at_admission", "boolean"), ("room_rent_deduction_pct", "double"),
                          ("copay_pct", "int"), ("reason_code", "string"), ("reasoning", "string")],
}
for _t, _exp in _EXPECTED.items():
    _got = [(f.name, f.dataType.simpleString()) for f in spark.table(_t).schema.fields]
    assert _got == _exp, (_t, _got)

_ver = {r["claim_id"]: r.asDict() for r in spark.table(VERIFIER_TABLE).collect()}
_adj = {r["claim_id"]: r.asDict() for r in spark.table(ADJUDICATIONS_TABLE).collect()}
assert len(_ver) == 20 and len(_adj) == 20
assert all(r["reason_code"] in REASON_CODES for r in _ver.values())
assert all(r["reason_code"] in REASON_CODES for r in _adj.values())

# room_rent_deduction_pct really is 2-decimal DOUBLE
for _t, _d in (("verifier", _ver), ("adjudications", _adj)):
    _bad = [cid for cid, r in _d.items()
            if abs(float(r["room_rent_deduction_pct"]) * 100 - round(float(r["room_rent_deduction_pct"]) * 100)) > 1e-6]
    assert not _bad, f"{_t} deduction not 2 decimals: {_bad}"

# reasoning >= 20 chars on >= 18 rows
_n_ok = sum(1 for r in _adj.values() if r["reasoning"] and len(r["reasoning"]) >= 20)
assert _n_ok >= 18, f"reasoning >= 20 chars on only {_n_ok}/20 rows"

# --- agreement recomputed independently in SQL (different engine than cell 4's python path) ---
_sql = spark.sql(f"""
  SELECT AVG(CASE WHEN v.payable = a.payable
                    AND v.waiting_period_ok = a.waiting_period_ok
                    AND v.network_at_admission = a.network_at_admission
                    AND abs(v.room_rent_deduction_pct - a.room_rent_deduction_pct) <= 0.01
                    AND v.copay_pct = a.copay_pct
                    AND v.reason_code = a.reason_code
             THEN 1.0 ELSE 0.0 END) AS agreement
  FROM {VERIFIER_TABLE} v
  INNER JOIN {ADJUDICATIONS_TABLE} a ON v.claim_id = a.claim_id
""").collect()[0]
_agreement = float(_sql["agreement"])
assert _agreement >= AGREEMENT_TARGET, _agreement

# --- code needles in the sql/ file ---
with open(SQL_NEEDLE_FILE) as f:
    _txt = f.read()
assert re.search(r"ai_query\s*\(", _txt, re.I), "needle missing: ai_query("
assert re.search(r"responseFormat\s*=>", _txt, re.I), "needle missing: responseFormat"
assert re.search(r"from_json\s*\(", _txt, re.I), "needle missing: from_json"
assert f"'{CHAT_MODEL}'" in _txt and re.search(r"'temperature'\s*,\s*0(?:\.0+)?\s*\)", _txt)

print(f"VERIFIED: probes 20 ids == csv | verifier 20x7 + adjudications 20x8, exact types "
      f"(boolean/boolean/boolean/double/int/string[/string]) | reason codes in enum | "
      f"deductions 2dp | reasoning >= 20 chars on {_n_ok}/20 | SQL agreement {_agreement:.0%} "
      f">= {AGREEMENT_TARGET:.0%} | needles present in {SQL_NEEDLE_FILE} ({CHAT_MODEL}, temperature=0)")
display(spark.sql(f"""
  SELECT v.claim_id, v.payable AS v_payable, a.payable AS ai_payable,
         v.reason_code AS v_reason, a.reason_code AS ai_reason,
         v.room_rent_deduction_pct AS v_ded, a.room_rent_deduction_pct AS ai_ded,
         v.copay_pct AS v_copay, a.copay_pct AS ai_copay
  FROM {VERIFIER_TABLE} v JOIN {ADJUDICATIONS_TABLE} a USING (claim_id)
  ORDER BY v.claim_id"""))

In [0]:
%sql
select * from claims.genai.adjudications;